# Pickleball: đề xuất pha bóng từ video toàn trận
Pipeline riêng cho góc quay cố định. Chưa có mô hình nhận diện bóng/điểm thắng. Ví dụ tổng hợp dưới đây kiểm tra phân đoạn và ngân sách, không chứng minh độ chính xác trên trận thật.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"pickleball.py").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
from pickleball import analyze_match, propose_rallies, choose_rallies
from core import export_video
from IPython.display import display, Video

## 1. Khoảng hoạt động thay cho clip cố định
Bắt đầu khi điểm hoạt động vượt ngưỡng; kết thúc sau khi mức thấp kéo dài hơn khoảng nghỉ cho phép. Giữ trước/sau 2 giây. Đoạn dài hơn 90 giây cần duyệt. Dữ liệu dưới mô phỏng hai pha có một khoảng tạm dừng ngắn.

In [2]:
t=np.arange(180)/3
activity=np.zeros(len(t))
for a,b in [(10,16),(17,25),(40,48)]: activity[(t>=a)&(t<b)]=.8
df=pd.DataFrame({"time":t,"activity":activity,"cut":False})
candidates=propose_rallies(df,60)
display(pd.DataFrame(candidates)[["start","end","score","needs_review"]])
px.line(df,x="time",y="activity").show()

,start,end,score,needs_review
0,8.0,27.0,0.685000,False
1,38.0,50.0,0.666667,False


In [3]:
selected=choose_rallies(candidates,30)
assert sum(s["end"]-s["start"] for s in selected)<=30
display(pd.DataFrame(selected)[["start","end","score"]])

,start,end,score
0,8.0,27.0,0.685


## 2. Chọn video thật trên máy
Đổi RUN_REAL_VIDEO thành True để mở hộp thoại chọn video. Với giao diện web chỉ cần tải video lên. Bước này có thể mất nhiều thời gian trên toàn trận. Đặt ROI theo tỷ lệ (trái, trên, phải, dưới) để giới hạn vùng sân.

In [4]:
RUN_REAL_VIDEO=False
ROI=(0.,0.,1.,1.)
BUDGET_SECONDS=180
if RUN_REAL_VIDEO:
    import tkinter as tk
    from tkinter.filedialog import askopenfilename
    dialog=tk.Tk(); dialog.withdraw()
    source=askopenfilename(title="Chọn video pickleball",filetypes=[("Video","*.mp4 *.mkv *.mov *.avi")])
    dialog.destroy()
    if source:
        out=ROOT/"runs"/"pickleball_notebook"
        features,meta,candidates=analyze_match(source,out,roi=ROI)
        chosen=choose_rallies(candidates,BUDGET_SECONDS)
        display(pd.DataFrame(candidates))
        if chosen:
            video=export_video(source,chosen,out)
            display(Video(str(video),embed=False))
        else: print("Không có ứng viên phù hợp; cần điều chỉnh hoặc duyệt lại.")

## 3. Đánh giá đúng phạm vi
Không gọi các ứng viên là rally đã được xác nhận. Chấm nhãn theo trận, tách validation/test; đo mất đầu/cuối pha và giữ nhầm khoảng nghỉ. So sánh cùng thời lượng: bản clip 4 giây cũ, bản activity mới, bản đã duyệt. Lưu tất cả tham số, thời gian xử lý, độ phân giải nguồn và cấu hình CPU. Camera pan, replay và chuyển động giữa các điểm vẫn có thể gây lỗi.